# 03 — Modelos y comparación en validación
### Proyecto: Predicción de duración de viajes — NYC Yellow Taxi (ene-abr 2026)

**Avance 1.** Entrenamos tres familias de modelos más los dos baselines, todos con el mismo
preprocesamiento (`src/features.py`) y la misma partición temporal:

| Conjunto | Meses | Uso |
|---|---|---|
| Train | enero, febrero | ajustar modelos |
| Validación | marzo | comparar modelos y tomar decisiones |
| Test | abril | evaluación única del modelo elegido |

El entrenamiento lo hace `src/train.py` (reproducible desde la terminal). Este notebook lo ejecuta y
lee sus salidas en `outputs/` y `reports/figures/` para analizarlas.

In [ ]:
import os, sys
sys.path.append('../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import data as D
from features import add_features, ALL_FEATURES, NUMERIC_FEATURES, ONEHOT_FEATURES, TARGET_ENC_FEATURES, TARGET

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
SEED = 42
np.random.seed(SEED)

# Ruta al parquet combinado (ver README). Se puede cambiar con la variable de entorno TAXI_DATA.
DATA_PATH = os.environ.get('TAXI_DATA', '../data/yellow_2026_jan_apr.parquet')
FIG = '../reports/figures'
os.makedirs(FIG, exist_ok=True)
import json
from IPython.display import Image, display
OUT = '../outputs'


## 1. Ejecutar el pipeline

Con ~4 GB de RAM se recomienda el muestreo por defecto (700k viajes por mes de train y 1M de marzo).
`--rapido` sirve para comprobar que todo corre en menos de un minuto.

In [ ]:
import subprocess
cmd = [sys.executable, '../src/train.py', '--etapa', 'validacion', '--data', DATA_PATH]
if os.environ.get('RAPIDO'):
    cmd.append('--rapido')
print(subprocess.run(cmd, capture_output=True, text=True, check=True).stdout)

## 2. Modelos comparados

| Nombre | Familia | Notas |
|---|---|---|
| `baseline_mediana` | constante | mediana de train |
| `baseline_lineal` | lineal | distancia + hora y día (one-hot). Versión corregida del baseline de la entrega previa |
| `ridge` | lineal regularizado | todas las features, estandarizadas |
| `random_forest` | bagging de árboles | 80 árboles, profundidad 20, 300k filas por árbol |
| `hgb_mae` | gradient boosting | pérdida absoluta (optimiza directamente el MAE) |
| `hgb_mse` | gradient boosting | pérdida cuadrática (optimiza el RMSE) |

Los hiperparámetros son valores razonables por defecto; la búsqueda sistemática es el siguiente paso.

In [ ]:
val = json.load(open(f'{OUT}/metrics_validacion.json'))
res = pd.DataFrame(val['resultados']).T
res['mejora_MAE_vs_lineal_%'] = (100 * (1 - res['MAE_min'] / res.loc['baseline_lineal', 'MAE_min'])).round(1)
res.sort_values('MAE_min')

In [ ]:
display(Image(f'{FIG}/07_comparacion_modelos_validacion.png'))

**Lectura:** completar con los números reales. Puntos a comentar:
- cuánto mejora cada familia sobre el baseline lineal y si la mejora es grande frente a la diferencia entre modelos;
- MAE vs. RMSE: `hgb_mae` debería ganar en MAE y `hgb_mse` en RMSE. Elegimos por **MAE** (métrica principal);
- costo de entrenamiento (`fit_seg`): el Random Forest es mucho más lento que HGB.

## 3. Sensibilidad a `trip_distance` (riesgo de leakage)

`trip_distance` es la distancia real recorrida, que en un caso de uso real se conocería solo como estimación.
Entrenamos el mismo HGB sin las dos variables de distancia:

In [ ]:
sens = val['sensibilidad_sin_distancia']
mejor = val['mejor_modelo']
pd.DataFrame({'con distancia (' + mejor + ')': val['resultados'][mejor], 'sin distancia (hgb_mae)': sens}).T[['MAE_min', 'RMSE_min', 'R2']]

**Lectura:** la diferencia entre las dos filas es la parte del desempeño que depende de conocer la distancia.
Si es muy grande, la versión "lista para producción" necesitaría una distancia estimada por un motor de rutas
(p. ej. distancia de red entre centroides de zona); se propone como trabajo del siguiente avance.

## 4. Predicho vs. real e importancia de variables

In [ ]:
display(Image(f'{FIG}/08_pred_vs_real_validacion.png'))

In [ ]:
imp = pd.read_csv(f'{OUT}/importancia_variables.csv')
display(imp.head(12))
display(Image(f'{FIG}/12_importancia_permutacion.png'))

La importancia por permutación mide cuánto empeora el MAE en validación al desordenar cada variable original
(antes del preprocesamiento), así que se lee directamente en minutos.

## 5. Primer análisis de errores por segmento

In [ ]:
seg = pd.read_csv(f'{OUT}/errores_por_segmento_validacion.csv')
for s in seg['segmento'].unique():
    print('\n==', s)
    display(seg[seg['segmento'] == s].drop(columns='segmento').set_index('valor'))

In [ ]:
for f in ['09_mae_por_distancia.png', '10_mae_por_duracion_real.png', '11_error_por_hora.png']:
    display(Image(f'{FIG}/{f}'))

**Lectura:** completar. Preguntas guía:
- ¿El error crece con la distancia en términos absolutos? ¿Y en relativos (MAPE)?
- ¿El modelo **subestima** los viajes largos (sesgo negativo)? Es lo esperable con pérdida MAE y cola larga.
- ¿Las horas pico tienen más error? (congestión no observada: clima, accidentes, eventos).
- ¿Hay diferencias por `VendorID` o por `RatecodeID` faltante? Eso sería un sesgo por proveedor a discutir.

## 6. Evaluación en test (abril)

Se ejecuta **una sola vez**, después de fijar el modelo con validación: se reentrena con enero-marzo y se evalúa en abril.
No se vuelve a tocar el modelo después de ver este número.

In [ ]:
cmd = [sys.executable, '../src/train.py', '--etapa', 'test', '--data', DATA_PATH]
if os.environ.get('RAPIDO'):
    cmd.append('--rapido')
print(subprocess.run(cmd, capture_output=True, text=True, check=True).stdout)
test = json.load(open(f'{OUT}/metrics_test.json'))
pd.DataFrame(test['resultados']).T